# OCR Gate 2 — local partial checkpoint audit

Run locally from the repository or its parent directory. The notebook auto-discovers the three recovered JSONLs, validates their exact checksums/IDs, and writes an operational audit beside them. It does not use GPU, call Gemini, or claim visual accuracy without human-labeled crops.


In [ ]:
import hashlib
import json
import os
import statistics
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

FILENAMES = ['easyocr-frames.jsonl', 'vintern-candidates.jsonl', 'vintern-results.jsonl']
EXPECTED = {
    'easyocr-frames.jsonl': (4164, 'e13a4fff27401f896af9f37ab180895ad5f35fb7dbaeb656e4c3882453842a46', 'keyframe_uid'),
    'vintern-candidates.jsonl': (34335, 'c54cb410f3fd1787d832a9713fbf069b098948a2f545f2c19720935e2f844ce9', 'candidate_id'),
    'vintern-results.jsonl': (27927, '5dd3a6eb772f2c3ca5b46125e0249403c0f9a360f167f33986bd5793b28f6c99', 'candidate_id'),
}
VIDEO_IDS = ['L21_V001', 'L21_V002', 'L21_V003', 'L21_V005', 'L21_V006']

configured = os.environ.get('OCR_GATE2_CHECKPOINT_DIR')
roots = [Path(configured)] if configured else []
roots.extend([Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent])
matches = [root.resolve() for root in roots if all((root / name).is_file() for name in FILENAMES)]
if not matches:
    raise FileNotFoundError('Set OCR_GATE2_CHECKPOINT_DIR to the directory containing the three recovered JSONLs.')
checkpoint_dir = matches[0]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

def load_jsonl(filename, id_field):
    rows, seen = [], set()
    with (checkpoint_dir / filename).open('r', encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            identity = row[id_field]
            if identity in seen:
                raise ValueError(f'duplicate {id_field} at line {line_number}: {identity}')
            seen.add(identity)
            rows.append(row)
    return rows

loaded, file_manifest = {}, {}
for filename, (expected_count, expected_hash, id_field) in EXPECTED.items():
    path = checkpoint_dir / filename
    actual_hash = sha256_file(path)
    if actual_hash != expected_hash:
        raise RuntimeError(f'{filename} checksum mismatch: {actual_hash}')
    rows = load_jsonl(filename, id_field)
    if len(rows) != expected_count:
        raise RuntimeError(f'{filename}: {len(rows)} != {expected_count}')
    loaded[filename] = rows
    file_manifest[filename] = {'records': len(rows), 'sha256': actual_hash}

easy = loaded['easyocr-frames.jsonl']
candidates = loaded['vintern-candidates.jsonl']
vintern = loaded['vintern-results.jsonl']
candidate_ids = {row['candidate_id'] for row in candidates}
result_ids = {row['candidate_id'] for row in vintern}
if not result_ids <= candidate_ids:
    raise RuntimeError(f'foreign Vintern result IDs: {len(result_ids - candidate_ids)}')

regions = sum(len(row.get('regions', [])) for row in easy)
candidate_fraction = len(candidates) / max(1, regions)
candidate_by_video = Counter(row['video_id'] for row in candidates)
result_by_video = Counter(row['video_id'] for row in vintern)
similarities = [row['text_similarity'] for row in vintern if isinstance(row.get('text_similarity'), (int, float))]
latencies = [row['inference_seconds'] for row in vintern if isinstance(row.get('inference_seconds'), (int, float))]

def percentile(values, fraction):
    values = sorted(values)
    return values[min(len(values) - 1, round((len(values) - 1) * fraction))] if values else None

coverage = {
    video_id: {
        'completed': result_by_video[video_id],
        'candidates': candidate_by_video[video_id],
        'fraction': result_by_video[video_id] / max(1, candidate_by_video[video_id]),
    }
    for video_id in VIDEO_IDS
}
mean_latency = statistics.fmean(latencies) if latencies else None
estimated_full_vintern_regions = round(293336 * len(candidates) / len(easy))
report = {
    'schema_version': 1,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'decision': 'FAIL_OPERATIONAL_FANOUT_PENDING_MANUAL_ACCURACY',
    'checkpoint_dir': str(checkpoint_dir),
    'files': file_manifest,
    'validation': {'json_valid': True, 'duplicate_ids': 0, 'foreign_result_ids': 0},
    'easyocr': {
        'frames': len(easy),
        'status': dict(Counter(row['status'] for row in easy)),
        'regions': regions,
        'regions_per_frame': regions / len(easy),
        'mixed_frames': sum(bool(row.get('frame_mixed_candidate')) for row in easy),
    },
    'router': {
        'vintern_candidates': len(candidates),
        'candidate_fraction_of_regions': candidate_fraction,
        'reasons': dict(Counter(reason for row in candidates for reason in row.get('escalation_reasons', []))),
        'estimated_full_catalog_vintern_regions': estimated_full_vintern_regions,
    },
    'vintern': {
        'completed': len(vintern),
        'unfinished': len(candidate_ids - result_ids),
        'status': dict(Counter(row['status'] for row in vintern)),
        'coverage_by_video': coverage,
        'latency_seconds_mean': mean_latency,
        'latency_seconds_p95': percentile(latencies, 0.95),
        'estimated_single_t4_hours_full_catalog': estimated_full_vintern_regions * mean_latency / 3600 if mean_latency else None,
    },
    'agreement_not_accuracy': {
        'easyocr_vintern_similarity_mean': statistics.fmean(similarities) if similarities else None,
        'similarity_p50': percentile(similarities, 0.50),
        'similarity_below_0_30': sum(value < 0.30 for value in similarities),
        'strong_disagreement_residuals': sum('strong_disagreement' in row.get('gemini_residual_reasons', []) for row in vintern),
    },
    'manual_accuracy': {
        'status': 'BLOCKED_MISSING_LOCAL_KEYFRAME_IMAGES',
        'reason': 'The recovered JSONLs contain Kaggle source paths; OCR correctness cannot be judged without the source crops or human ground truth.',
    },
}
output = checkpoint_dir / 'ocr_gate2_partial_auto_audit.json'
output.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))
print('AUDIT_WRITTEN', output)
